In [1]:
import os
OPENAI_API_KEY = os.environ['OPENAI_API_KEY']

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

In [2]:
cornwall_granular_collection = Chroma(
    collection_name="cornwall_granular",
    embedding_function=OpenAIEmbeddings(api_key=OPENAI_API_KEY)
)

In [3]:
cornwall_granular_collection.reset_collection()

In [4]:
cornwall_coarse_collection = Chroma(
    collection_name="cornwall_coarse",
    embedding_function=OpenAIEmbeddings(api_key=OPENAI_API_KEY)
)
cornwall_granular_collection.reset_collection()

In [5]:
os.environ["USER_AGENT"] = "manning-ch08/1.0 (mic.a.elle.chlon@gmail.com)"

from langchain_community.document_loaders import AsyncHtmlLoader
destination_url = "https://en.wikivoyage.org/wiki/Cornwall"
html_loader = AsyncHtmlLoader(destination_url)
docs = html_loader.load()

Fetching pages: 100%|######################################################################################################################################| 1/1 [00:00<00:00,  4.94it/s]


In [6]:
from langchain_text_splitters import HTMLSectionSplitter

header_to_split_on = [("h1", "Header 1"), ("h2", "Header 2")]
html_section_splitter = HTMLSectionSplitter(
    headers_to_split_on=header_to_split_on
)

def split_docs_into_granular_chunks(docs):
    all_chunks = []
    for doc in docs:
        html_string = doc.page_content
        temp_chunks = html_section_splitter.split_text(
            html_string
        )
        all_chunks.extend(temp_chunks)

    return all_chunks

In [7]:
granular_chunks = split_docs_into_granular_chunks(docs)

cornwall_granular_collection.add_documents(documents=granular_chunks)

['74aad997-c501-45f1-b648-261528542b7c',
 '6c80759a-3fa0-406e-9f52-dbdf240c86ff',
 '0fa9d49a-7a9e-4cc4-adf3-afc3736ae9fc',
 '7bab6607-bb87-4db6-aa5d-0edcb0025b0b',
 '0964cae1-e220-49d1-8733-c877ad7a6961',
 'f345b665-651c-46bf-ab22-184ecfae27a5',
 '434b3be3-ab4f-48ab-9592-9327ced2fd50',
 'fdf13465-b2c0-41f1-aad5-312d1cb7895c',
 '16460c04-fcfe-4825-aa8d-8ccb1c520c2b',
 'b6146d3a-2215-4400-8ed2-f93eaa362531',
 'a81e000f-4edb-425f-954d-b7547e54df1d',
 '55282231-9d31-4ada-9e06-16e32972fcfa',
 '2cec2168-9526-4448-82c2-77043c5d30ff',
 '4e6ca95b-d7a2-4a14-bd77-05ec46f78d1b',
 'f439680b-fc46-4ecc-8f9f-0065f13bc67a',
 'ec308d0f-c9d7-463d-9c34-367badb291e9',
 'c144b237-1269-46c7-9b1c-c7c42a15597f',
 'ab9fa002-4be2-4d75-93ce-3fdede9532d9',
 '741550bb-00e8-4b94-9657-13c30a03264a']

In [8]:
results = cornwall_granular_collection.similarity_search(
    query="Events or festivals in Cornwall",k=3
)

for doc in results:
    print(doc)

page_content='Cornwall' metadata={'Header 1': 'Cornwall'}
page_content='Festivals 
 [ edit ] 
 
 These festivals tend to not be public holidays and not all are celebrated fully across the county. 
   
 AberFest .   A Celtic cultural festival celebrating “All things” Cornish and Breton that takes place biennially (every two years) in Cornwall at Easter. The AberFest Festival alternates with the Breizh – Kernow Festival that is held in Brandivy and Bignan (in Breizh/Bretagne – France) on the alternate years.       ( updated Jun 2023 ) 
 Golowan , sometimes also  Goluan  or  Gol-Jowan  is the Cornish word for the Midsummer celebrations, most popular in the Penwith area and in particular  Penzance  and  Newlyn . The celebrations are conducted from the 23rd of June (St John's Eve) to the 28th of June (St Peter's Eve) each year, St Peter's Eve being the more popular in Cornish fishing communities. The celebrations are centred around the lighting of bonfires and fireworks and the performance 

In [9]:
from langchain_community.document_transformers import Html2TextTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

html2text_transformer = Html2TextTransformer()
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000, chunk_overlap=300
)

In [10]:
def split_docs_into_coarse_chunks(docs):
    text_docs = html2text_transformer.transform_documents(docs)
    coarse_chunks = text_splitter.split_documents(text_docs)
    return coarse_chunks

In [12]:
coarse_chunks = split_docs_into_coarse_chunks(docs)

cornwall_coarse_collection.add_documents(documents=coarse_chunks)

['e75b5d4d-7dc8-4d22-8e4a-cf006d98e0e2',
 '6eac7f12-7da5-4473-83cb-4aac38b79b2a',
 '1bf6c83d-5edb-42f6-b28a-c4fddf83f772',
 '6d9804b2-1972-47da-999e-af89a6d74684',
 '41be17f9-b0ba-40fe-86ea-1469bc0cbf06',
 'eb1054ef-bff0-44e7-95ea-c37a39e64035',
 '4d17c4b4-0af2-4eb3-b574-50a79f70cb08',
 'd48a3f08-13ce-40ed-87c4-ec5f5c9ce081',
 '277844a6-a038-458a-88fd-a0498666c27d',
 '73700f01-1ae8-4942-9e07-7d8d77f90f6a',
 '1db1ff77-5e43-4236-9d4e-83f1b972f9b4',
 '5a987191-25fb-4301-9700-451ff9adc376',
 'd5707df4-b4c0-4788-9c68-d513f1c3b218',
 '2540883c-b757-43ef-9079-07df13d678e0',
 '3904ce00-54f8-40e3-b9ea-4492d19499d9']

In [13]:
results = cornwall_coarse_collection.similarity_search(
    query="Events or Festival in Cornwall", k=3
)

for doc in results:
    print(doc)

page_content='### Spirits

[edit]

    _See also:Liquor_

Gin and rum are also produced in Cornwall. A popular brand of Cornish rum is
Dead Man's Fingers which has multiple flavours and is bottled in St. Ives.

## Festivals

[edit]

These festivals tend to not be public holidays and not all are celebrated
fully across the county.

AberFest. A Celtic cultural festival celebrating “All things” Cornish and
Breton that takes place biennially (every two years) in Cornwall at Easter.
The AberFest Festival alternates with the Breizh – Kernow Festival that is
held in Brandivy and Bignan (in Breizh/Bretagne – France) on the alternate
years. (updated Jun 2023)

**Golowan** , sometimes also _Goluan_ or _Gol-Jowan_ is the Cornish word for
the Midsummer celebrations, most popular in the Penwith area and in particular
Penzance and Newlyn. The celebrations are conducted from the 23rd of June (St
John's Eve) to the 28th of June (St Peter's Eve) each year, St Peter's Eve
being the more popular in Corni

In [14]:
uk_granular_collection = Chroma(
    collection_name="uk_granular",
    embedding_function=OpenAIEmbeddings(api_key=OPENAI_API_KEY),
)
uk_granular_collection.reset_collection()

uk_coarse_collection = Chroma(
    collection_name="uk_coarse",
    embedding_function=OpenAIEmbeddings(api_key=OPENAI_API_KEY),
)
uk_coarse_collection.reset_collection()

uk_destinations = [
    "Cornwall", "North_Cornwall", "South_Cornwall", "West_Cornwall",
    "Tintagel", "Bodmin", "Wadebridge", "Penzance", "Newquay",
    "St_Ives", "Port_Isaac", "Looe", "Polperro", "Porthleven",
    "East_Sussex", "Brighton", "Battle", "Hastings_(England)",
    "Rye_(England)", "Seaford", "Ashdown_Forest"
]

wikivoyage_root_url = "https://en.wikivoyage.org/wiki"
uk_destination_urls = [f'{wikivoyage_root_url}/{d}' 
                       for d in uk_destinations]

for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url)
    docs = html_loader.load()

granular_chunks = split_docs_into_granular_chunks(docs)
uk_granular_collection.add_documents(documents=granular_chunks)

coarse_chunks = split_docs_into_coarse_chunks(docs)
uk_coarse_collection.add_documents(documents=coarse_chunks)

Fetching pages: 100%|######################################################################################################################################| 1/1 [00:00<00:00,  5.56it/s]


['47053636-78f0-4a1b-ad4b-696913330872',
 'ae3c1a43-5215-4029-ae35-443128c299fb',
 '80c890b2-e624-413d-94d5-2a2bf6461a55',
 '5ee8e6f0-966d-4d6f-bc71-000f4decac3a',
 'ec479b40-de3d-410a-9105-ba5716bdd29f',
 'f0e733a7-98c6-4f23-857c-7576e0087f29',
 '5d738872-6f98-4d48-9bfa-b9582e824148',
 '7da800fb-18ea-442b-9ffc-b058cb020071',
 'e8f3031a-620a-4e7b-b859-733cbc804743']

In [15]:
granular_results = uk_granular_collection.similarity_search(
    query="Events or festivals in East Sussex",k=4)

for doc in granular_results:
    print(doc)
    print("\n------------------------------------------------------------------\n")

coarse_results = uk_coarse_collection.similarity_search(
    query="Events or festivals in East Sussex",k=4)

for doc in coarse_results:
    print(doc)

page_content='Go next 
 [ edit ] 
 
 
 
 Royal Tunbridge Wells  (on the A26) - Victorian spa town with bars, pubs and drinking fountains for the local water. 
 Eastbourne 
 Petersfield 
 
 South Downs Way , a popular walking path. 
 
 London , a train ride away. 
 Kent 
 Medway 
 Crowborough 
 
 
 
 
 Routes through Ashdown Forest 
 
 
 
 
 
 
 
 
 London   ←   East Grinstead   ← 
 
 
   N     S   
 
 
 →   Uckfield   →   Eastbourne 
 
 
 
 
 .mw-parser-output .routeBox{font-size:small;border-style:none;border-spacing:0 0;border-collapse:collapse;margin:0 auto}.mw-parser-output .routeBox td{padding:1px 2px} 
 
 
 
 
 
 
 
 .mw-parser-output .article-status{width:60%;background:#fff;color:black;margin:0 auto;border:solid 2px lightblue;text-align:center;font-size:90%;font-style:italic}.mw-parser-output .article-status-disambig{border:2px dashed lightblue}.mw-parser-output .article-status-disambig td:first-child{width:48px;text-align:center}.mw-parser-output .article-status-stub{border:1p